# Tutorial: 3-Node AdEx Mean-Field PhiID Sweep

This notebook builds a minimal `3`-node Zerlaut AdEx mean-field system and uses pairwise bivariate PhiID to turn each simulated condition into `3 x 3` `STS` and `RTR` matrices.

## Network setup

- node `1` and node `2` are structurally coupled
- node `3` is structurally uncoupled
- the structural matrix is fixed, and the global coupling parameter `G` scales the effective coupling strength
- private OU noise is varied via `weight_noise`

## Simulation settings

- mean-field timescale `T = 20 ms`
- simulation duration `5000 ms`
- transient removed `1000 ms`
- analysis window `4000 ms`
- integration step `dt = 0.1 ms`
- raw monitor period `1 ms`

## PhiID idea

For each simulated condition, we take the `3` excitatory-rate time series and compute bivariate PhiID for each pair:

- pair `(1,2)`
- pair `(1,3)`
- pair `(2,3)`

Those pairwise values are written back into symmetric `3 x 3` matrices for:

- synergy `STS`
- redundancy `RTR`

and this is done separately for `MMI` and `CCS`.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import pandas as pd

REPO = Path('/Users/borjan/CNRS/projects/TVBToolkit').resolve()
if str(REPO / 'src') not in sys.path:
    sys.path.insert(0, str(REPO / 'src'))

from tvbtoolkit.analysis import (
    load_three_node_phiid_index,
    plot_three_node_hypothesis_summary,
    plot_three_node_matrix_grid,
    summarize_three_node_outputs,
)

OUTPUT_ROOT = REPO / 'results' / 'three_node_adex_phiid_sweep'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT

## Launch the sweep

The full runner simulates the AdEx system in Python, exports each condition, calls MATLAB to compute `MMI` and `CCS` PhiID matrices, and then renders the figures.

Suggested first run:

- `G` sweep: `0.0 0.05 0.1 0.2 0.3 0.4`
- noise sweep: `0.0 2.5e-5 5e-5 7.5e-5 1e-4 1.25e-4 1.5e-4 2e-4 3e-4`
- seeds: `1 2 3 4 5 6`
- first-order Zerlaut for speed (`zerlaut_order = 1`)

You can increase seeds or switch to second-order later for robustness.

In [ ]:
cmd = [
    sys.executable,
    str(REPO / 'scripts' / 'run_three_node_adex_phiid_sweep.py'),
    '--output-root', str(OUTPUT_ROOT),
    '--g-values', '0.0', '0.05', '0.1', '0.2', '0.3', '0.4',
    '--noise-values', '0.0', '2.5e-5', '5e-5', '7.5e-5', '1e-4', '1.25e-4', '1.5e-4', '2e-4', '3e-4',
    '--seeds', '1', '2', '3', '4', '5', '6',
    '--zerlaut-order', '1',
    '--python-workers', '1',
    '--run-matlab',
    '--matlab-parallel',
    '--matlab-workers', '4',
]
# subprocess.run(cmd, check=True)
cmd

## Load the outputs

In [ ]:
manifest_path = OUTPUT_ROOT / 'manifest.csv'
index_df = load_three_node_phiid_index(OUTPUT_ROOT / 'phiid', manifest_path=manifest_path)
raw_df, avg_df = summarize_three_node_outputs(index_df)
raw_df.head()

## Matrix grids

Each panel is a full `3 x 3` PhiID matrix for one `(G, noise)` condition.

In [ ]:
g_values = sorted(avg_df['g_value'].unique().tolist())
noise_values = sorted(avg_df['noise_value'].unique().tolist())

fig, _ = plot_three_node_matrix_grid(avg_df, measure='mmi', atom='sts', g_values=g_values, noise_values=noise_values)
fig

In [ ]:
fig, _ = plot_three_node_matrix_grid(avg_df, measure='mmi', atom='rtr', g_values=g_values, noise_values=noise_values)
fig

In [ ]:
fig, _ = plot_three_node_matrix_grid(avg_df, measure='ccs', atom='sts', g_values=g_values, noise_values=noise_values)
fig

In [ ]:
fig, _ = plot_three_node_matrix_grid(avg_df, measure='ccs', atom='rtr', g_values=g_values, noise_values=noise_values)
fig

## Hypothesis-oriented summaries

These panels collapse the `3 x 3` matrices into the pairs most relevant to the hypothesis:

- `STS`: average of mixed pairs `(1,3)` and `(2,3)`
- `RTR`: coupled pair `(1,2)`

In [ ]:
fig, _ = plot_three_node_hypothesis_summary(avg_df)
fig

## Interpretation notes

The intended qualitative hypothesis is:

- redundancy should be strongest for the structurally coupled pair `(1,2)`
- synergy may be stronger for the mixed pairs `(1,3)` and `(2,3)` if indirect or less-locked interactions become more important than direct overlap

Whether that appears in `MMI`, `CCS`, or both is an empirical result of the sweep, not something assumed by the code.